# Machine Learning Drug Recommendation System using Python and Streamlit

This notebook demonstrates an end-to-end educational drug recommendation project using:

- Synthetic patient data
- Data preprocessing
- Random Forest classification
- Model evaluation
- Prediction probabilities
- Rule-based safety checks
- Model export for Streamlit deployment

> **Medical disclaimer:** This project is for education and research only. It must not be used to diagnose disease, prescribe medication, select dosage, or replace licensed clinical judgment.


## 1. Import required libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Generate a synthetic healthcare dataset

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

condition_drugs = {
    "Hypertension": ["Lisinopril", "Amlodipine", "Losartan"],
    "Type 2 Diabetes": ["Metformin", "Glipizide", "Sitagliptin"],
    "High Cholesterol": ["Atorvastatin", "Rosuvastatin", "Simvastatin"],
    "Asthma": ["Albuterol", "Budesonide", "Montelukast"],
    "Acid Reflux": ["Omeprazole", "Pantoprazole", "Famotidine"],
}

rows = []

for condition, candidate_drugs in condition_drugs.items():
    for _ in range(250):
        age = int(rng.integers(18, 86))
        sex = rng.choice(["Female", "Male"])
        bmi = round(float(rng.uniform(18, 42)), 1)
        systolic_bp = int(rng.integers(95, 181))
        glucose = int(rng.integers(70, 261))
        cholesterol = int(rng.integers(120, 321))
        kidney_disease = int(rng.random() < 0.12)
        liver_disease = int(rng.random() < 0.08)
        penicillin_allergy = int(rng.random() < 0.10)
        pregnant = int(sex == "Female" and age < 50 and rng.random() < 0.08)

        # Simplified synthetic labeling rules
        if condition == "Hypertension":
            if age >= 60:
                recommended_drug = "Amlodipine"
            elif kidney_disease == 0:
                recommended_drug = "Losartan"
            else:
                recommended_drug = "Lisinopril"

        elif condition == "Type 2 Diabetes":
            if kidney_disease == 0:
                recommended_drug = "Metformin"
            elif age > 65:
                recommended_drug = "Sitagliptin"
            else:
                recommended_drug = "Glipizide"

        elif condition == "High Cholesterol":
            if cholesterol > 250:
                recommended_drug = "Rosuvastatin"
            elif age < 70:
                recommended_drug = "Atorvastatin"
            else:
                recommended_drug = "Simvastatin"

        elif condition == "Asthma":
            if age < 45:
                recommended_drug = "Albuterol"
            elif age < 70:
                recommended_drug = "Budesonide"
            else:
                recommended_drug = "Montelukast"

        else:
            if liver_disease:
                recommended_drug = "Famotidine"
            elif age > 60:
                recommended_drug = "Pantoprazole"
            else:
                recommended_drug = "Omeprazole"

        # Add realistic label noise for demonstration
        if rng.random() < 0.10:
            recommended_drug = rng.choice(candidate_drugs)

        rows.append({
            "age": age,
            "sex": sex,
            "bmi": bmi,
            "systolic_bp": systolic_bp,
            "glucose": glucose,
            "cholesterol": cholesterol,
            "kidney_disease": kidney_disease,
            "liver_disease": liver_disease,
            "penicillin_allergy": penicillin_allergy,
            "pregnant": pregnant,
            "condition": condition,
            "recommended_drug": recommended_drug,
        })

df = pd.DataFrame(rows)

print("Dataset shape:", df.shape)
df.head()


In [ ]:
# Save the generated dataset
DATA_PATH = Path("drug_recommendation_dataset.csv")
df.to_csv(DATA_PATH, index=False)
print(f"Dataset saved to: {DATA_PATH.resolve()}")


## 3. Explore the dataset

In [ ]:
df.info()


In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))

print("\nTarget distribution:")
display(df["recommended_drug"].value_counts().to_frame("count"))


In [ ]:
condition_counts = df["condition"].value_counts()

plt.figure(figsize=(9, 5))
condition_counts.plot(kind="bar")
plt.title("Patient Records by Condition")
plt.xlabel("Condition")
plt.ylabel("Number of Records")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 4. Prepare features and target

In [ ]:
FEATURES = [
    "age",
    "sex",
    "bmi",
    "systolic_bp",
    "glucose",
    "cholesterol",
    "kidney_disease",
    "liver_disease",
    "penicillin_allergy",
    "pregnant",
    "condition",
]

TARGET = "recommended_drug"

X = df[FEATURES]
y = df[TARGET]

categorical_features = ["sex", "condition"]
numerical_features = [feature for feature in FEATURES if feature not in categorical_features]

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)


## 5. Split the dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 6. Build the preprocessing and Random Forest pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        (
            "numerical",
            "passthrough",
            numerical_features,
        ),
    ]
)

classifier = RandomForestClassifier(
    n_estimators=300,
    max_depth=14,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ]
)

model_pipeline


## 7. Train the model

In [ ]:
model_pipeline.fit(X_train, y_train)
print("Model training completed.")


## 8. Evaluate the model

In [ ]:
y_pred = model_pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy:.2%}")


In [ ]:
report = classification_report(
    y_test,
    y_pred,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).T
report_df.round(3)


In [ ]:
labels = sorted(y.unique())

cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(14, 12))
display_plot = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels,
)
display_plot.plot(
    ax=ax,
    xticks_rotation=90,
    values_format="d",
)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


## 9. Create a prediction function

In [ ]:
def predict_drug(patient_data: dict, top_k: int = 5):
    patient_df = pd.DataFrame([patient_data])

    predicted_drug = model_pipeline.predict(patient_df)[0]
    probabilities = model_pipeline.predict_proba(patient_df)[0]
    classes = model_pipeline.classes_

    ranking = (
        pd.DataFrame({
            "drug": classes,
            "probability": probabilities,
        })
        .sort_values("probability", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

    return predicted_drug, ranking


In [ ]:
sample_patient = {
    "age": 58,
    "sex": "Female",
    "bmi": 29.4,
    "systolic_bp": 152,
    "glucose": 108,
    "cholesterol": 215,
    "kidney_disease": 0,
    "liver_disease": 0,
    "penicillin_allergy": 0,
    "pregnant": 0,
    "condition": "Hypertension",
}

prediction, ranking = predict_drug(sample_patient)

print("Predicted medication class:", prediction)
display(ranking.style.format({"probability": "{:.2%}"}))


## 10. Add rule-based safety checks

In [ ]:
DRUG_INFORMATION = {
    "Lisinopril": {
        "purpose": "ACE inhibitor commonly used for hypertension.",
        "common_side_effects": "Dry cough, dizziness, and increased potassium.",
    },
    "Amlodipine": {
        "purpose": "Calcium-channel blocker commonly used for hypertension.",
        "common_side_effects": "Ankle swelling, flushing, and dizziness.",
    },
    "Losartan": {
        "purpose": "Angiotensin receptor blocker commonly used for hypertension.",
        "common_side_effects": "Dizziness and increased potassium.",
    },
    "Metformin": {
        "purpose": "Medicine commonly used as first-line treatment for type 2 diabetes.",
        "common_side_effects": "Nausea, diarrhea, and abdominal discomfort.",
    },
    "Glipizide": {
        "purpose": "Sulfonylurea used to lower blood glucose.",
        "common_side_effects": "Low blood sugar and weight gain.",
    },
    "Sitagliptin": {
        "purpose": "DPP-4 inhibitor used for type 2 diabetes.",
        "common_side_effects": "Headache and upper respiratory symptoms.",
    },
    "Atorvastatin": {
        "purpose": "Statin used to reduce LDL cholesterol.",
        "common_side_effects": "Muscle pain and elevated liver enzymes.",
    },
    "Rosuvastatin": {
        "purpose": "High-potency statin used to reduce LDL cholesterol.",
        "common_side_effects": "Muscle pain and headache.",
    },
    "Simvastatin": {
        "purpose": "Statin used to reduce LDL cholesterol.",
        "common_side_effects": "Muscle pain and digestive discomfort.",
    },
    "Albuterol": {
        "purpose": "Short-acting bronchodilator used for quick asthma relief.",
        "common_side_effects": "Tremor, rapid heartbeat, and nervousness.",
    },
    "Budesonide": {
        "purpose": "Inhaled corticosteroid used for asthma control.",
        "common_side_effects": "Oral thrush and hoarseness.",
    },
    "Montelukast": {
        "purpose": "Leukotriene receptor antagonist used for asthma.",
        "common_side_effects": "Headache; mood or behavior changes require review.",
    },
    "Omeprazole": {
        "purpose": "Proton-pump inhibitor used for acid reflux.",
        "common_side_effects": "Headache and abdominal discomfort.",
    },
    "Pantoprazole": {
        "purpose": "Proton-pump inhibitor used for acid reflux.",
        "common_side_effects": "Headache and diarrhea.",
    },
    "Famotidine": {
        "purpose": "H2 blocker used to reduce stomach acid.",
        "common_side_effects": "Headache, dizziness, and constipation.",
    },
}


In [ ]:
def safety_review(drug: str, patient: dict):
    warnings = []

    if patient.get("pregnant", 0):
        if drug in {
            "Lisinopril",
            "Losartan",
            "Atorvastatin",
            "Rosuvastatin",
            "Simvastatin",
        }:
            warnings.append(
                f"{drug} may be inappropriate during pregnancy. "
                "Immediate clinician review is required."
            )

    if patient.get("kidney_disease", 0):
        if drug == "Metformin":
            warnings.append(
                "Kidney function must be evaluated before metformin use."
            )

        if drug in {
            "Lisinopril",
            "Losartan",
            "Sitagliptin",
            "Famotidine",
        }:
            warnings.append(
                f"{drug} may require renal dose adjustment or monitoring."
            )

    if patient.get("liver_disease", 0):
        if drug in {
            "Atorvastatin",
            "Rosuvastatin",
            "Simvastatin",
        }:
            warnings.append(
                "Statin therapy requires clinician assessment and liver monitoring."
            )

    if patient.get("age", 0) >= 65:
        warnings.append(
            "Older adults may require lower starting doses and closer monitoring."
        )

    if patient.get("systolic_bp", 120) >= 180:
        warnings.append(
            "Systolic blood pressure at or above 180 mmHg may require urgent evaluation."
        )

    if patient.get("glucose", 100) >= 250:
        warnings.append(
            "Very high glucose may require prompt clinical assessment."
        )

    return warnings


In [ ]:
predicted_drug, ranking = predict_drug(sample_patient)
warnings = safety_review(predicted_drug, sample_patient)

print("Predicted drug:", predicted_drug)
print("Drug information:", DRUG_INFORMATION[predicted_drug])

if warnings:
    print("\nSafety warnings:")
    for warning in warnings:
        print("-", warning)
else:
    print("\nNo rule-based warning was triggered.")
    print("A licensed clinician must still review the recommendation.")


## 11. Save the trained model and project metadata

In [ ]:
MODEL_PATH = Path("drug_recommendation_model.joblib")
METRICS_PATH = Path("drug_recommendation_metrics.json")

joblib.dump(model_pipeline, MODEL_PATH)

metrics = {
    "accuracy": float(accuracy),
    "training_rows": int(len(X_train)),
    "testing_rows": int(len(X_test)),
    "features": FEATURES,
    "classes": sorted(y.unique().tolist()),
}

METRICS_PATH.write_text(json.dumps(metrics, indent=2))

print(f"Model saved to: {MODEL_PATH.resolve()}")
print(f"Metrics saved to: {METRICS_PATH.resolve()}")


## 12. Streamlit application code

Save the following code in a separate file named `app.py` in the same folder as the exported model.


In [ ]:
streamlit_code = r"""
from pathlib import Path
import joblib
import pandas as pd
import streamlit as st

MODEL_PATH = Path("drug_recommendation_model.joblib")

st.set_page_config(
    page_title="Drug Recommendation System",
    page_icon="💊",
    layout="wide",
)

st.title("💊 Machine Learning Drug Recommendation System")
st.warning(
    "Educational prototype only. It does not diagnose, prescribe, "
    "or replace a licensed healthcare professional."
)

@st.cache_resource
def load_model():
    return joblib.load(MODEL_PATH)

model = load_model()

with st.sidebar:
    st.header("Patient Information")

    age = st.slider("Age", 18, 90, 45)
    sex = st.selectbox("Sex", ["Female", "Male"])
    condition = st.selectbox(
        "Primary condition",
        [
            "Hypertension",
            "Type 2 Diabetes",
            "High Cholesterol",
            "Asthma",
            "Acid Reflux",
        ],
    )

    bmi = st.number_input("BMI", 12.0, 60.0, 26.0, step=0.1)
    systolic_bp = st.number_input(
        "Systolic blood pressure",
        70,
        250,
        125,
    )
    glucose = st.number_input(
        "Glucose",
        40,
        500,
        105,
    )
    cholesterol = st.number_input(
        "Total cholesterol",
        80,
        500,
        190,
    )

    kidney_disease = st.checkbox("Kidney disease")
    liver_disease = st.checkbox("Liver disease")
    penicillin_allergy = st.checkbox("Penicillin allergy")
    pregnant = st.checkbox(
        "Pregnant",
        disabled=(sex == "Male"),
    )

    predict_button = st.button(
        "Generate recommendation",
        type="primary",
        use_container_width=True,
    )

if predict_button:
    patient = {
        "age": age,
        "sex": sex,
        "bmi": bmi,
        "systolic_bp": systolic_bp,
        "glucose": glucose,
        "cholesterol": cholesterol,
        "kidney_disease": int(kidney_disease),
        "liver_disease": int(liver_disease),
        "penicillin_allergy": int(penicillin_allergy),
        "pregnant": int(
            pregnant if sex == "Female" else False
        ),
        "condition": condition,
    }

    patient_df = pd.DataFrame([patient])

    prediction = model.predict(patient_df)[0]
    probabilities = model.predict_proba(patient_df)[0]

    probability_df = (
        pd.DataFrame({
            "Drug": model.classes_,
            "Probability": probabilities,
        })
        .sort_values("Probability", ascending=False)
        .head(5)
    )

    st.success(f"Suggested medication class: **{prediction}**")
    st.metric(
        "Model confidence",
        f"{probability_df.iloc[0]['Probability']:.1%}",
    )

    st.subheader("Top model probabilities")
    st.dataframe(
        probability_df.style.format({
            "Probability": "{:.2%}"
        }),
        use_container_width=True,
    )

    st.subheader("Patient profile")
    st.dataframe(
        patient_df,
        use_container_width=True,
    )
else:
    st.info(
        "Enter patient information in the sidebar and "
        "select Generate recommendation."
    )
"""

Path("app.py").write_text(streamlit_code)
print("Streamlit app saved as app.py")


## 13. Run the Streamlit application

Open a terminal in the notebook folder and run:

```bash
pip install streamlit pandas numpy scikit-learn joblib matplotlib
streamlit run app.py
```

The application normally opens at:

```text
http://localhost:8501
```


## 14. Research and production improvements

A real clinical decision-support system would require:

- De-identified and properly governed clinical datasets
- Medication interaction and contraindication databases
- Dosage and renal/hepatic adjustment logic
- External and prospective clinical validation
- Fairness and subgroup evaluation
- Explainability and audit trails
- Privacy and access controls
- Human clinician approval
- Regulatory and institutional review
- Post-deployment safety monitoring
